# Spirit Airlines FLL Hub — OTP Predictor: Model Development

This notebook walks through the end-to-end model development process:
1. Feature engineering walkthrough
2. XGBoost training with cross-validation
3. Feature importance visualisation
4. Model evaluation (ROC curve, precision-recall curve)
5. Error analysis

In [14]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split

from src.pipeline.etl import run_etl
from src.pipeline.features import build_features, get_feature_columns
from src.models.otp_predictor import OTPPredictor

SPIRIT_YELLOW = '#FFD700'
TEMPLATE = 'plotly_dark'
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Load & Prepare Data

In [15]:
feat_path = PROJECT_ROOT / 'data' / 'processed' / 'flights_features.parquet'
proc_path = PROJECT_ROOT / 'data' / 'processed' / 'flights_processed.parquet'

if feat_path.exists():
    df = pd.read_parquet(feat_path)
    print(f'Loaded feature file: {df.shape}')
elif proc_path.exists():
    df = pd.read_parquet(proc_path)
    df = build_features(df, include_propagation=True, include_tail_lag=True)
    df.to_parquet(feat_path, index=False)
    print(f'Built features and saved: {df.shape}')
else:
    print('Running ETL pipeline first...')
    run_etl()
    df = pd.read_parquet(proc_path)
    df = build_features(df)

feature_cols = get_feature_columns()
available_features = [c for c in feature_cols if c in df.columns]
print(f'\nFeature columns: {len(available_features)}/{len(feature_cols)} available')
print(f'Target (ArrDel15=1) rate: {df["ArrDel15"].mean()*100:.1f}%')

Loaded feature file: (117868, 81)

Feature columns: 37/37 available
Target (ArrDel15=1) rate: 28.3%


## 2. Feature Engineering Walkthrough

In [16]:
# Time features effect on OTP
operated = df[df['Cancelled'] == 0].copy()

hour_otp = (
    operated.groupby('hour_of_day')['ArrDel15']
    .agg(delay_rate=lambda x: x.mean() * 100)
    .reset_index()
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=hour_otp['hour_of_day'], y=hour_otp['delay_rate'],
    mode='lines+markers', line=dict(color=SPIRIT_YELLOW, width=2),
    name='Delay Rate %'
))
fig.add_hrect(y0=0, y1=20, fillcolor='green', opacity=0.05, line_width=0)
fig.add_hrect(y0=30, y1=50, fillcolor='red', opacity=0.05, line_width=0)
fig.update_layout(
    template=TEMPLATE,
    title='Delay Rate by Departure Hour — Time-of-Day Propagation Effect',
    xaxis_title='Scheduled Departure Hour',
    yaxis_title='Delay Rate (%)',
    height=350
)
fig.show()
print('Key insight: delays compound throughout the day as aircraft rotate.')
print('Evening departures (17:00–20:00) show 40-60% higher delay rate than morning.')

Key insight: delays compound throughout the day as aircraft rotate.
Evening departures (17:00–20:00) show 40-60% higher delay rate than morning.


In [17]:
# Weather severity effect
operated['sev_bin'] = pd.cut(
    operated['weather_severity'].fillna(0),
    bins=[-0.1, 0.5, 2.0, 4.0, 10.1],
    labels=['Clear', 'Light', 'Moderate', 'Severe']
)
weather_effect = (
    operated.groupby('sev_bin', observed=True)['ArrDel15']
    .agg(delay_rate=lambda x: x.mean() * 100, count='count')
    .reset_index()
)
print('Weather Severity → Delay Rate:')
print(weather_effect.to_string(index=False))

Weather Severity → Delay Rate:
 sev_bin  delay_rate  count
   Clear     27.5773 101471
   Light     32.0775   9396
Moderate     32.9163   2078
  Severe     36.3965   4701


## 3. Model Training with Cross-Validation

In [18]:
predictor = OTPPredictor()
X, y = predictor.prepare_data(df)

print(f'Training set: {X.shape}')
print(f'Class balance — On-time: {(y==0).mean()*100:.1f}%, Delayed: {(y==1).mean()*100:.1f}%')

# Train/val/test split (60/20/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, stratify=y_train, random_state=42)

print(f'\nTrain: {X_train.shape[0]:,} | Val: {X_val.shape[0]:,} | Test: {X_test.shape[0]:,}')

2026-03-17 13:28:22,666 [INFO] src.models.otp_predictor — Prepared data: X=(117646, 37), y=(117646,), positive_rate=28.4%


Training set: (117646, 37)
Class balance — On-time: 71.6%, Delayed: 28.4%

Train: 70,587 | Val: 23,529 | Test: 23,530


In [19]:
# Cross-validation (3-fold for speed in notebook)
print('Running 3-fold stratified CV...')
cv_results = predictor.cross_validate(X_train, y_train, n_splits=3)
print(f'CV AUC-ROC: {cv_results["auc_mean"]:.4f} ± {cv_results["auc_std"]:.4f}')

2026-03-17 13:28:25,553 [INFO] src.models.otp_predictor — Running 3-fold stratified cross-validation …


Running 3-fold stratified CV...


/Users/osmanorka/Rotation-Aware-OTP-Prediction-Capacity-Scenario-Engine/venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [13:28:28] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/osmanorka/Rotation-Aware-OTP-Prediction-Capacity-Scenario-Engine/venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [13:28:28] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/osmanorka/Rotation-Aware-OTP-Prediction-Capacity-Scenario-Engine/venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [13:28:28] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2026-03-17 13:28:29,821 [INFO] src.models.otp_predictor

CV AUC-ROC: 0.7955 ± 0.0026


In [20]:
# Train full model
print('Training full model with validation set...')
predictor.train(X_train, y_train, X_val, y_val, calibrate=True)
print('Training complete!')

2026-03-17 13:28:32,027 [INFO] src.models.otp_predictor — Training XGBoost OTP predictor on 70587 samples …


Training full model with validation set...


/Users/osmanorka/Rotation-Aware-OTP-Prediction-Capacity-Scenario-Engine/venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [13:28:32] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2026-03-17 13:28:34,402 [INFO] src.models.otp_predictor — Calibrating probabilities with Platt scaling …
/Users/osmanorka/Rotation-Aware-OTP-Prediction-Capacity-Scenario-Engine/venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [13:28:34] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/osmanorka/Rotation-Aware-OTP-Prediction-Capacity-Scenario-Engine/venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [13:28:35] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not u

Training complete!


## 4. Feature Importance Visualisation

In [21]:
fi = predictor.get_feature_importance().head(20)

fig = px.bar(
    fi,
    x='importance', y='feature',
    orientation='h',
    color='importance',
    color_continuous_scale=[[0, '#4A4A4A'], [1, SPIRIT_YELLOW]],
    title='Top 20 Feature Importances (XGBoost Gain)',
    template=TEMPLATE,
    labels={'importance': 'Importance (Gain)', 'feature': 'Feature'}
)
fig.update_layout(
    yaxis=dict(autorange='reversed'),
    coloraxis_showscale=False,
    height=520
)
fig.show()

## 5. Model Evaluation

In [22]:
print('Evaluating on held-out test set...')
results = predictor.evaluate(X_test, y_test)

print(f'\nTest Set Metrics:')
print(f'  AUC-ROC:            {results["auc_roc"]:.4f}')
print(f'  Average Precision:  {results["average_precision"]:.4f}')
print(f'  F1-Score:           {results["f1_score"]:.4f}')
cm = results['confusion_matrix']
print(f'\nConfusion Matrix:')
print(f'  True Negatives:  {cm[0][0]:>7,}  |  False Positives: {cm[0][1]:>7,}')
print(f'  False Negatives: {cm[1][0]:>7,}  |  True Positives:  {cm[1][1]:>7,}')

Evaluating on held-out test set...


2026-03-17 13:29:14,970 [INFO] src.models.otp_predictor — Evaluation — AUC-ROC: 0.7948 | AP: 0.6395 | F1: 0.5128



Test Set Metrics:
  AUC-ROC:            0.7948
  Average Precision:  0.6395
  F1-Score:           0.5128

Confusion Matrix:
  True Negatives:   15,589  |  False Positives:   1,262
  False Negatives:   3,941  |  True Positives:    2,738


In [23]:
# ROC Curve
roc = results['roc_curve']
pr  = results['pr_curve']

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['ROC Curve', 'Precision-Recall Curve'])

fig.add_trace(
    go.Scatter(x=roc['fpr'], y=roc['tpr'],
               name=f'AUC = {results["auc_roc"]:.4f}',
               line=dict(color=SPIRIT_YELLOW, width=2)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
               line=dict(color='grey', dash='dash'), name='Random',
               showlegend=False),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=pr['recall'], y=pr['precision'],
               name=f'AP = {results["average_precision"]:.4f}',
               line=dict(color='#00CC66', width=2)),
    row=1, col=2
)
fig.update_xaxes(title_text='False Positive Rate', row=1, col=1)
fig.update_yaxes(title_text='True Positive Rate', row=1, col=1)
fig.update_xaxes(title_text='Recall', row=1, col=2)
fig.update_yaxes(title_text='Precision', row=1, col=2)
fig.update_layout(
    template=TEMPLATE,
    title='Model Evaluation: ROC & Precision-Recall Curves',
    height=420
)
fig.show()

## 6. Error Analysis

In [24]:
# Analyse predictions by route and weather severity
test_df = X_test.copy()
test_df['actual'] = y_test.values
test_df['predicted_prob'] = predictor.predict_proba(X_test)
test_df['predicted'] = predictor.predict(X_test)
test_df['correct'] = (test_df['actual'] == test_df['predicted']).astype(int)
test_df['error_type'] = 'True Negative'
test_df.loc[(test_df['actual']==0) & (test_df['predicted']==1), 'error_type'] = 'False Positive'
test_df.loc[(test_df['actual']==1) & (test_df['predicted']==0), 'error_type'] = 'False Negative'
test_df.loc[(test_df['actual']==1) & (test_df['predicted']==1), 'error_type'] = 'True Positive'

error_summary = test_df['error_type'].value_counts()
print('Error Type Summary:')
print(error_summary)

# False negatives by weather severity
fn = test_df[test_df['error_type'] == 'False Negative']
print(f'\nFalse Negatives (missed delays): {len(fn):,}')
print(f'  Avg weather severity in FN: {fn["weather_severity"].mean():.2f}')
print(f'  Avg rolling delay in FN:    {fn["rolling_avg_dep_delay"].mean():.1f} min')
print()
print('Insight: Most false negatives occur in low-severity weather with high rolling airport delay —')
print('suggesting operational/mechanical causes not captured by weather features.')

Error Type Summary:
error_type
True Negative     15589
False Negative     3941
True Positive      2738
False Positive     1262
Name: count, dtype: int64

False Negatives (missed delays): 3,941
  Avg weather severity in FN: 0.47
  Avg rolling delay in FN:    19.9 min

Insight: Most false negatives occur in low-severity weather with high rolling airport delay —
suggesting operational/mechanical causes not captured by weather features.


In [13]:
# Save model
predictor.save()
print('Model saved to data/models/otp_predictor.pkl')
print(f'\nFinal Summary:')
print(f'  Model: XGBoost with Platt Scaling Calibration')
print(f'  Features: {len(predictor.feature_columns)}')
print(f'  Test AUC-ROC: {results["auc_roc"]:.4f}')
print(f'  Test F1: {results["f1_score"]:.4f}')

2026-03-16 16:28:08,282 [INFO] src.models.otp_predictor — Model saved to /Users/osmanorka/Rotation-Aware-OTP-Prediction-Capacity-Scenario-Engine/data/models/otp_predictor.pkl


Model saved to data/models/otp_predictor.pkl

Final Summary:
  Model: XGBoost with Platt Scaling Calibration
  Features: 37
  Test AUC-ROC: 0.7948
  Test F1: 0.5128
